# Stage 5 (Extra Credit) — Hierarchical Script-Routed Classifier

The flat 23-way OvR model from Stage 3/4 scored **0.9150 test macro-F1**, but the confusion
matrix showed the errors were not spread evenly — they were almost entirely concentrated in
one cluster: the 8 Devanagari-script languages (`brx_Deva`, `doi_Deva`, `gom_Deva`, `hin_Deva`,
`mai_Deva`, `mar_Deva`, `npi_Deva`, `san_Deva`), where `gom_Deva` (Konkani) alone accounted for
79 misclassifications, nearly all of them predicted as `mar_Deva` (Marathi).

This notebook builds a **two-stage hierarchical classifier** aimed directly at that finding,
instead of a generic tweak:

1. **Stage A — script router.** A sentence's dominant Unicode script block is computed directly
   (no ML, no training data needed). For 10 of the 23 languages, script alone already uniquely
   identifies the language, so routing is the final answer for those — free accuracy.
2. **Stage B — small within-bucket classifiers** for the two easy multi-language buckets
   (`asm_Beng`/`ben_Beng`, and `kas_Arab`/`snd_Deva`/`urd_Arab`), reusing the existing global
   TF-IDF features from Stage 2 — these buckets were not the problem, so no extra machinery is
   spent on them.
3. **Stage C — a specialized classifier for the one hard bucket** (the 8 Devanagari languages),
   with two upgrades over the global model: character n-grams extended out to **5-grams**, and
   vanilla IDF replaced by a **supervised, class-discriminative term weighting (tf-chi²)** — a
   term's weight now reflects how well it separates *these 8 specific languages from each other*,
   not just how rare it is across the whole 23-language corpus. This is the "richer features
   spent just where they're needed" idea, applied literally.

This mirrors, in a fully hand-built form, what real closely-related-language-identification
research does: 2025's CHiPSAL workshop (COLING 2025) ran a shared task on exactly this Devanagari
cluster (Hindi/Marathi/Nepali/Bhojpuri/Sanskrit), evaluated on macro-F1, because generic
whole-vocabulary classifiers consistently struggle right where we struggled.


In [1]:
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

DATA_DIR = "../data"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

X_train_g = sp.load_npz(f"{DATA_DIR}/X_train.npz").tocsr()
X_val_g = sp.load_npz(f"{DATA_DIR}/X_val.npz").tocsr()
X_test_g = sp.load_npz(f"{DATA_DIR}/X_test.npz").tocsr()

with open(f"{DATA_DIR}/languages.json", encoding="utf-8") as f:
    ALL_LANGUAGES = json.load(f)
baseline_W = np.load(f"{DATA_DIR}/model_W.npy")
baseline_b = np.load(f"{DATA_DIR}/model_b.npy")

print("train/val/test:", train_df.shape, val_df.shape, test_df.shape)
print("global feature dims:", X_train_g.shape, X_val_g.shape, X_test_g.shape)


train/val/test: (18400, 2) (2300, 2) (2300, 2)
global feature dims: (18400, 214895) (2300, 214895) (2300, 214895)


## Stage A — the script router

No model, no training: for each sentence, count which Unicode script block its characters
mostly fall into, and call that the "bucket." This reuses the same script-range logic from the
Stage 1 sanity check, now used as an actual routing decision rather than a diagnostic.


In [2]:
SCRIPT_RANGES = {
    "Deva": (0x0900, 0x097f), "Beng": (0x0980, 0x09ff), "Gujr": (0x0a80, 0x0aff),
    "Guru": (0x0a00, 0x0a7f), "Orya": (0x0b00, 0x0b7f), "Taml": (0x0b80, 0x0bff),
    "Telu": (0x0c00, 0x0c7f), "Knda": (0x0c80, 0x0cff), "Mlym": (0x0d00, 0x0d7f),
    "Arab": (0x0600, 0x06ff), "Mtei": (0xabc0, 0xabff), "Olck": (0x1c50, 0x1c7f),
}


def dominant_script(text):
    counts = {}
    for ch in text:
        cp = ord(ch)
        matched = None
        for name, (lo, hi) in SCRIPT_RANGES.items():
            if lo <= cp <= hi:
                matched = name
                break
        if matched is None and ch.isalpha():
            matched = "Latn"
        if matched:
            counts[matched] = counts.get(matched, 0) + 1
    if not counts:
        return "Latn"
    return max(counts, key=counts.get)


# ground-truth script for each language label, used only for diagnostics below
LANGUAGE_SCRIPT = {
    "asm_Beng": "Beng", "ben_Beng": "Beng",
    "brx_Deva": "Deva", "doi_Deva": "Deva", "gom_Deva": "Deva", "hin_Deva": "Deva",
    "mai_Deva": "Deva", "mar_Deva": "Deva", "npi_Deva": "Deva", "san_Deva": "Deva",
    "guj_Gujr": "Gujr", "kan_Knda": "Knda", "mal_Mlym": "Mlym", "mni_Mtei": "Mtei",
    "ory_Orya": "Orya", "pan_Guru": "Guru", "tam_Taml": "Taml", "tel_Telu": "Telu",
    "kas_Arab": "Arab", "snd_Deva": "Arab", "urd_Arab": "Arab",
    "khasi": "Latn", "santhali": "Olck",
}

BUCKETS = {}
for lang, script in LANGUAGE_SCRIPT.items():
    BUCKETS.setdefault(script, []).append(lang)

for script, langs in sorted(BUCKETS.items(), key=lambda kv: -len(kv[1])):
    suffix = "s" if len(langs) != 1 else ""
    print(f"{script:6s} ({len(langs)} language{suffix}): {langs}")


Deva   (8 languages): ['brx_Deva', 'doi_Deva', 'gom_Deva', 'hin_Deva', 'mai_Deva', 'mar_Deva', 'npi_Deva', 'san_Deva']
Arab   (3 languages): ['kas_Arab', 'snd_Deva', 'urd_Arab']
Beng   (2 languages): ['asm_Beng', 'ben_Beng']
Gujr   (1 language): ['guj_Gujr']
Knda   (1 language): ['kan_Knda']
Mlym   (1 language): ['mal_Mlym']
Mtei   (1 language): ['mni_Mtei']
Orya   (1 language): ['ory_Orya']
Guru   (1 language): ['pan_Guru']
Taml   (1 language): ['tam_Taml']
Telu   (1 language): ['tel_Telu']
Latn   (1 language): ['khasi']
Olck   (1 language): ['santhali']


### Diagnostic: how accurate is the router, before any classifier even runs?

This number is a hard ceiling on the whole hierarchical system — no downstream classifier can
fix a sentence that got routed to the wrong bucket.


In [3]:
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pred_bucket = df["text"].apply(dominant_script)
    true_bucket = df["label"].map(LANGUAGE_SCRIPT)
    acc = (pred_bucket == true_bucket).mean()
    n_wrong = (pred_bucket != true_bucket).sum()
    print(f"{split_name:5s}: router bucket accuracy = {acc:.4f}  ({n_wrong} misrouted / {len(df)})")


train: router bucket accuracy = 0.9942  (107 misrouted / 18400)
val  : router bucket accuracy = 0.9943  (13 misrouted / 2300)
test : router bucket accuracy = 0.9930  (16 misrouted / 2300)


## Stage B — small classifiers for the easy multi-language buckets

`Beng` (`asm_Beng` vs `ben_Beng`) and `Arab` (`kas_Arab` vs `snd_Deva` vs `urd_Arab`) were never
the problem in the flat model — the baseline confusion matrix shows these already near-perfect.
So no new features here: these two buckets simply reuse the existing global TF-IDF matrices from
Stage 2, restricted to their own rows, with a small One-vs-Rest classifier — the exact same
classifier design as Stage 3, just trained on fewer classes.


In [4]:
def sigmoid(z):
    out = np.empty_like(z, dtype=np.float64)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


class OneVsRestLogisticRegression:
    """Same design as Stage 3: K independent binary logistic regressions, trained together
    via one vectorized full-batch gradient descent loop. No sklearn."""

    def __init__(self, languages, lr=0.5, reg=10.0, epochs=350):
        self.languages = list(languages)
        self.lr = lr
        self.reg = reg
        self.epochs = epochs
        self.W = None
        self.b = None

    def _one_hot(self, y_labels):
        idx = {lang: i for i, lang in enumerate(self.languages)}
        Y = np.zeros((len(y_labels), len(self.languages)))
        for i, lab in enumerate(y_labels):
            Y[i, idx[lab]] = 1.0
        return Y

    def fit(self, X, y_labels):
        n, v = X.shape
        k = len(self.languages)
        self.W = np.zeros((v, k))
        self.b = np.zeros(k)
        Y = self._one_hot(y_labels)
        Xt = X.T.tocsr()
        for _ in range(self.epochs):
            Z = X.dot(self.W) + self.b
            P = sigmoid(Z)
            G = P - Y
            gW = Xt.dot(G) / n + (self.reg / n) * self.W
            gb = G.mean(axis=0)
            self.W -= self.lr * gW
            self.b -= self.lr * gb
        return self

    def predict_proba(self, X):
        return sigmoid(X.dot(self.W) + self.b)

    def predict(self, X):
        P = self.predict_proba(X)
        return np.array(self.languages)[P.argmax(axis=1)]


In [5]:
def train_bucket_from_global_features(bucket_languages, label="bucket"):
    mask_tr = train_df["label"].isin(bucket_languages).values
    mask_va = val_df["label"].isin(bucket_languages).values

    Xb_tr = X_train_g[mask_tr]
    yb_tr = train_df["label"].values[mask_tr]
    Xb_va = X_val_g[mask_va]
    yb_va = val_df["label"].values[mask_va]

    model = OneVsRestLogisticRegression(bucket_languages).fit(Xb_tr, yb_tr)
    train_acc = np.mean(model.predict(Xb_tr) == yb_tr)
    val_acc = np.mean(model.predict(Xb_va) == yb_va)
    print(f"{label}: train rows={Xb_tr.shape[0]}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")
    return model


beng_model = train_bucket_from_global_features(BUCKETS["Beng"], "Beng bucket")
arab_model = train_bucket_from_global_features(BUCKETS["Arab"], "Arab bucket")


Beng bucket: train rows=1600  train_acc=0.9969  val_acc=0.9900


Arab bucket: train rows=2400  train_acc=0.9925  val_acc=0.9933


## Stage C — the hard bucket: `Deva`, with richer features

Here we actually spend the extra effort. Two changes from the global Stage 2 vectorizer, both
scoped only to this bucket's own 6,400 training sentences:

1. **Character n-grams extended to 5-grams** (global model used 2/3/4-grams only) — a wider
   receptive field to catch longer morphological markers that distinguish these closely related
   languages.
2. **tf-chi² term weighting instead of IDF.** IDF only asks "is this gram rare across the whole
   corpus?" — it has no idea whether a gram actually separates Marathi from Konkani specifically.
   Chi-square (a standard statistical test of independence between "does this gram appear" and
   "is this the correct class") answers exactly that question, per class. Each gram's final
   weight is the *maximum* chi-square score across the 8 languages — how discriminative it is
   for its single best-fit language — log-dampened the same way IDF is, to stop a few extreme
   values from dominating.


In [6]:
DEVA_LANGS = BUCKETS["Deva"]
MIN_DF_DEVA = 2


def tokenize_extended(text):
    """Same as Stage 2's tokenizer, but char n-grams go out to 5 instead of 4."""
    text = text.strip()
    words = text.split()
    grams = []
    for w in words:
        grams.append("W1|" + w)
    for i in range(len(words) - 1):
        grams.append("W2|" + words[i] + " " + words[i + 1])
    for n in (2, 3, 4, 5):
        for i in range(len(text) - n + 1):
            grams.append(f"C{n}|" + text[i:i + n])
    return grams


class ChiSquareNgramVectorizer:
    """Same n-gram + sparse + L2-normalize pipeline as Stage 2's TF-IDF vectorizer, but the
    per-gram weight is a supervised tf-chi2 score instead of IDF. No sklearn."""

    def __init__(self, languages, min_df=2):
        self.languages = list(languages)
        self.min_df = min_df
        self.vocab_ = None
        self.weight_ = None

    def fit(self, texts, labels):
        df_per_class = {lang: Counter() for lang in self.languages}
        df_total = Counter()
        n_per_class = {lang: 0 for lang in self.languages}

        for text, label in zip(texts, labels):
            grams = set(tokenize_extended(text))
            df_per_class[label].update(grams)
            df_total.update(grams)
            n_per_class[label] += 1

        n_docs = len(texts)
        vocab_terms = sorted(g for g, c in df_total.items() if c >= self.min_df)
        self.vocab_ = {g: i for i, g in enumerate(vocab_terms)}

        weight = np.zeros(len(vocab_terms))
        for j, term in enumerate(vocab_terms):
            dft = df_total[term]
            best_chi2 = 0.0
            for lang in self.languages:
                a = df_per_class[lang][term]              # term present, in class
                nk = n_per_class[lang]
                b = dft - a                                 # term present, not in class
                c = nk - a                                   # term absent, in class
                d = (n_docs - dft) - (nk - a)                # term absent, not in class
                denom = (a + b) * (c + d) * (a + c) * (b + d)
                if denom > 0:
                    chi2 = n_docs * (a * d - b * c) ** 2 / denom
                    if chi2 > best_chi2:
                        best_chi2 = chi2
            weight[j] = np.log1p(best_chi2)

        self.weight_ = weight
        return self

    def transform(self, texts):
        rows, cols, data = [], [], []
        for row_idx, text in enumerate(texts):
            tf = Counter(tokenize_extended(text))
            vec = {}
            for gram, count in tf.items():
                j = self.vocab_.get(gram)
                if j is not None:
                    vec[j] = count * self.weight_[j]
            norm = np.sqrt(sum(v * v for v in vec.values())) or 1.0
            for j, v in vec.items():
                rows.append(row_idx)
                cols.append(j)
                data.append(v / norm)
        return sp.csr_matrix((data, (rows, cols)), shape=(len(texts), len(self.vocab_)))

    def fit_transform(self, texts, labels):
        return self.fit(texts, labels).transform(texts)


In [7]:
deva_train_df = train_df[train_df["label"].isin(DEVA_LANGS)].reset_index(drop=True)
deva_val_df = val_df[val_df["label"].isin(DEVA_LANGS)].reset_index(drop=True)
deva_test_df = test_df[test_df["label"].isin(DEVA_LANGS)].reset_index(drop=True)

t0 = time.time()
deva_vectorizer = ChiSquareNgramVectorizer(DEVA_LANGS, min_df=MIN_DF_DEVA)
X_deva_train = deva_vectorizer.fit_transform(deva_train_df["text"].tolist(), deva_train_df["label"].tolist())
t1 = time.time()
X_deva_val = deva_vectorizer.transform(deva_val_df["text"].tolist())
X_deva_test = deva_vectorizer.transform(deva_test_df["text"].tolist())
t2 = time.time()

print(f"fit_transform(deva train) in {t1 - t0:.1f}s -> vocab = {len(deva_vectorizer.vocab_):,}")
print(f"transform(deva val, test) in {t2 - t1:.1f}s")
print("X_deva_train:", X_deva_train.shape, " X_deva_val:", X_deva_val.shape, " X_deva_test:", X_deva_test.shape)


fit_transform(deva train) in 3.0s -> vocab = 148,676
transform(deva val, test) in 0.3s
X_deva_train: (6400, 148676)  X_deva_val: (800, 148676)  X_deva_test: (800, 148676)


In [8]:
inv_vocab_deva = {i: g for g, i in deva_vectorizer.vocab_.items()}
top_idx = np.argsort(-deva_vectorizer.weight_)[:12]
print("Most class-discriminative grams inside the Deva bucket (highest tf-chi2 weight):")
for idx in top_idx:
    print(f"  {inv_vocab_deva[idx]:12s}  weight={deva_vectorizer.weight_[idx]:.3f}")


Most class-discriminative grams inside the Deva bucket (highest tf-chi2 weight):
  C3|नि         weight=7.917
  C3|नाय        weight=7.848
  C3|मा         weight=7.777
  C3| है        weight=7.754
  C3|को         weight=7.743
  C2|खौ         weight=7.710
  C3|दों        weight=7.640
  C2|है         weight=7.609
  C3|खौ         weight=7.603
  C3|के         weight=7.582
  C4| के        weight=7.573
  W1|के         weight=7.571


### A tuning note worth keeping honest

The first version of this cell reused Stage 3's exact hyperparameters (`lr=0.5, reg=10,
epochs=350`) and produced a suspiciously low **81% train accuracy** — worse than expected for a
classifier that should be able to at least fit its own training data reasonably well.
The likely cause: `reg` was left at the same nominal value, but `n` dropped from 18,400 (full
corpus) to 6,400 (this bucket only) — since the regularization pull in the gradient is `reg/n`,
the same `reg=10` is now roughly **3x stronger relative to the data** than it was in Stage 3.
A quick sweep confirmed this was underfitting, not a real ceiling: validation accuracy climbed
from 78.9% to 84.8% as `reg` came down and `epochs` went up, with a still-healthy (~4 point)
train/val gap. `lr=1.5, reg=0.5, epochs=2000` is used below.


In [9]:
t0 = time.time()
deva_model = OneVsRestLogisticRegression(DEVA_LANGS, lr=1.5, reg=0.5, epochs=2000).fit(
    X_deva_train, deva_train_df["label"].values)
t1 = time.time()

train_acc = np.mean(deva_model.predict(X_deva_train) == deva_train_df["label"].values)
val_acc = np.mean(deva_model.predict(X_deva_val) == deva_val_df["label"].values)
print(f"Deva bucket classifier trained in {t1 - t0:.1f}s")
print(f"train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")


Deva bucket classifier trained in 37.7s
train_acc=0.8856  val_acc=0.8475


## Stage D — full pipeline inference on the test set

For every test sentence: route by script. If the bucket has only one possible language, that's
the answer, for free. Otherwise dispatch to the matching Stage B/C classifier — Beng/Arab reuse
the already-computed global test features; Deva re-vectorizes with the bucket-specific
chi²-weighted vectorizer.


In [10]:
TRIVIAL_BUCKET_LANG = {script: langs[0] for script, langs in BUCKETS.items() if len(langs) == 1}
print("Trivial (free) buckets:", TRIVIAL_BUCKET_LANG)

test_router_bucket = test_df["text"].apply(dominant_script).values
hier_pred = np.empty(len(test_df), dtype=object)

for script, lang in TRIVIAL_BUCKET_LANG.items():
    mask = test_router_bucket == script
    hier_pred[mask] = lang

mask_beng = test_router_bucket == "Beng"
if mask_beng.any():
    hier_pred[mask_beng] = beng_model.predict(X_test_g[mask_beng])

mask_arab = test_router_bucket == "Arab"
if mask_arab.any():
    hier_pred[mask_arab] = arab_model.predict(X_test_g[mask_arab])

mask_deva = test_router_bucket == "Deva"
if mask_deva.any():
    deva_texts = test_df["text"].values[mask_deva].tolist()
    X_deva_route_test = deva_vectorizer.transform(deva_texts)
    hier_pred[mask_deva] = deva_model.predict(X_deva_route_test)

assert not (hier_pred == None).any()  # noqa: E711 -- every row must have been routed somewhere
print("all", len(hier_pred), "test rows assigned a prediction")


Trivial (free) buckets: {'Gujr': 'guj_Gujr', 'Knda': 'kan_Knda', 'Mlym': 'mal_Mlym', 'Mtei': 'mni_Mtei', 'Orya': 'ory_Orya', 'Guru': 'pan_Guru', 'Taml': 'tam_Taml', 'Telu': 'tel_Telu', 'Latn': 'khasi', 'Olck': 'santhali'}


all 2300 test rows assigned a prediction


## Evaluation — same from-scratch macro-F1 machinery as Stage 4

In [11]:
def confusion_matrix(y_true, y_pred, languages):
    idx = {lang: i for i, lang in enumerate(languages)}
    k = len(languages)
    cm = np.zeros((k, k), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[idx[t], idx[p]] += 1
    return cm


def precision_recall_f1_per_class(cm):
    k = cm.shape[0]
    precision = np.zeros(k)
    recall = np.zeros(k)
    f1 = np.zeros(k)
    for i in range(k):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision[i] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall[i] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1[i] = (2 * precision[i] * recall[i] / (precision[i] + recall[i])
                 if (precision[i] + recall[i]) > 0 else 0.0)
    return precision, recall, f1


def macro_f1_report(y_true, y_pred, languages):
    cm = confusion_matrix(y_true, y_pred, languages)
    precision, recall, f1 = precision_recall_f1_per_class(cm)
    return {"confusion_matrix": cm, "precision": precision, "recall": recall,
            "f1": f1, "macro_f1": f1.mean()}


In [12]:
y_test = test_df["label"].values

# baseline: the flat 23-way model from Stage 3/4, recomputed here for a like-for-like comparison
baseline_pred = np.array(ALL_LANGUAGES)[sigmoid(X_test_g.dot(baseline_W) + baseline_b).argmax(axis=1)]
baseline_report = macro_f1_report(y_test, baseline_pred, ALL_LANGUAGES)

hier_report = macro_f1_report(y_test, hier_pred, ALL_LANGUAGES)

print(f"Baseline (flat 23-way)     test macro-F1: {baseline_report['macro_f1']:.4f}")
print(f"Hierarchical (script-routed) test macro-F1: {hier_report['macro_f1']:.4f}")


Baseline (flat 23-way)     test macro-F1: 0.9150
Hierarchical (script-routed) test macro-F1: 0.9433


In [13]:
compare_df = pd.DataFrame({
    "language": ALL_LANGUAGES,
    "baseline_f1": baseline_report["f1"],
    "hierarchical_f1": hier_report["f1"],
})
compare_df["delta"] = compare_df["hierarchical_f1"] - compare_df["baseline_f1"]
compare_df = compare_df.sort_values("delta")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
compare_df


,language,baseline_f1,hierarchical_f1,delta
17,san_Deva,0.9746,0.9659,-0.0088
1,ben_Beng,0.9950,0.9901,-0.0049
12,mar_Deva,0.6884,0.6837,-0.0047
7,kan_Knda,1.0000,1.0000,0.0000
11,mal_Mlym,1.0000,1.0000,0.0000
13,mni_Mtei,0.9529,0.9529,0.0000
8,kas_Arab,1.0000,1.0000,0.0000
5,guj_Gujr,1.0000,1.0000,0.0000
19,snd_Deva,0.9950,0.9950,0.0000
22,urd_Arab,0.9950,0.9950,0.0000


## Honest error analysis

Two separate error sources feed into the hierarchical model's remaining mistakes: sentences
misrouted at Stage A (an unavoidable ceiling, since no downstream classifier can rescue a
sentence sent to the wrong bucket), and sentences routed correctly but still misclassified by
the Deva-bucket classifier itself.


In [14]:
true_bucket_test = test_df["label"].map(LANGUAGE_SCRIPT).values
misrouted = test_router_bucket != true_bucket_test
misrouted_and_wrong = misrouted & (hier_pred != y_test)
routed_ok_but_wrong = (~misrouted) & (hier_pred != y_test)

print(f"Total test errors: {(hier_pred != y_test).sum()} / {len(y_test)}")
print(f"  caused by Stage A misrouting:            {misrouted_and_wrong.sum()}")
print(f"  caused by a within-bucket classifier miss: {routed_ok_but_wrong.sum()}")

deva_mask_true = true_bucket_test == "Deva"
deva_cm = confusion_matrix(y_test[deva_mask_true & ~misrouted],
                            hier_pred[deva_mask_true & ~misrouted], DEVA_LANGS)
print("\nWithin-bucket confusion matrix for the 8 Deva languages (rows=true, cols=predicted):")
print(pd.DataFrame(deva_cm, index=DEVA_LANGS, columns=DEVA_LANGS))


Total test errors: 129 / 2300
  caused by Stage A misrouting:            16
  caused by a within-bucket classifier miss: 113

Within-bucket confusion matrix for the 8 Deva languages (rows=true, cols=predicted):
          brx_Deva  doi_Deva  gom_Deva  hin_Deva  mai_Deva  mar_Deva  \
brx_Deva        78         1         1        18         1         0   
doi_Deva         0        88         0         5         6         0   
gom_Deva         0         2        63         0         1        29   
hin_Deva         0         0         1        97         0         0   
mai_Deva         1         2         1         0        93         0   
mar_Deva         0         1        28         0         2        67   
npi_Deva         0         0         1         0         1         0   
san_Deva         0         0         0         0         0         0   

          npi_Deva  san_Deva  
brx_Deva         0         0  
doi_Deva         0         1  
gom_Deva         2         1  
hin_Deva        

## Summary

The hierarchical, script-routed design directly targets the failure mode the baseline's own
confusion matrix diagnosed, rather than a generic hyperparameter tweak: 10 of 23 languages are
resolved by script alone at zero classifier cost, two small multi-language buckets reuse the
existing global features unchanged, and the one genuinely hard cluster gets a dedicated
classifier with a wider character-gram window and supervised, class-discriminative term
weighting in place of plain IDF. The comparison table above shows exactly where — and by how
much — this moved the needle relative to the flat Stage 3/4 baseline.
